# L60 · autograd 机制 — grad_fn 计算图（computation graph），backward()，retain_grad 与梯度累积

**目标**：用 PyTorch autograd 对同一个计算图求梯度（gradient），并与 `L56_backward_pass.ipynb` 中手写的 `Value` 引擎数值比对，确认两套机制在数值上完全等价。

🔗 **Aurora 连接**：Month 2–5 的所有模型训练都调用 `.backward()`；本节确立的等价性是后续用 torch 替换手写 Value 图的信心基础。具体入口：`src/aurora/` 各核的 `trainer.py`（Month 3 起引入）中每个 `loss.backward()` 调用均依赖此机制。

← **上一课**　[L59 · PyTorch Tensor 基础](L59_tensor_basics.ipynb)

> 上节课学习了 **PyTorch Tensor 基础**：与 NumPy 互转、device、requiresgrad。  
> 本课将探讨 **autograd 机制**。

## 本课剧情：PyTorch 怎么"记住"你做了什么运算？

上节课自己实现了 `Value` 类——每个节点存着 `_backward` 闭包和 `_prev` 集合，手动建图。PyTorch 做的事完全一样，只是规模更大、速度更快。

**核心机制**：每个 `requires_grad=True` 的 Tensor 自动挂一个 `grad_fn`（梯度函数），记录"我是怎么算出来的"：

```python
a = torch.tensor(2.0, requires_grad=True)   # 叶节点，grad_fn=None
b = torch.tensor(3.0, requires_grad=True)   # 叶节点
c = a * b                                    # c.grad_fn = MulBackward0
d = c + a                                    # d.grad_fn = AddBackward0
```

图长这样：
```
a ──┬──[×]──► c ──[+]──► d
b ──┘         a ──────────┘
```

**叶节点 vs 非叶节点**：

| 类型 | 定义 | `.grad` 规则 |
|---|---|---|
| 叶节点 | `requires_grad=True` 且非运算结果 | backward后自动填入 `.grad` |
| 非叶节点 | 由运算创建 | `.grad=None`（除非 `retain_grad()`）|

**为什么非叶节点默认不存梯度**：中间激活值可能有数百万个，全存会爆显存。只保留叶节点（参数）的梯度，够训练用了。

本节任务：验证 `L = 2*a*b + 3*b` 对 `a, b` 的梯度，并理解 `detach()`、`retain_grad()`、梯度累积。

In [ ]:
import torch
import numpy as np

## 1. 计算图：`requires_grad=True` 与叶 / 非叶节点（non-leaf node）

当张量带 `requires_grad=True` 时，每次运算会生成一个 `grad_fn` 并把结果节点挂在图上。
**叶节点（leaf tensor）**：由用户直接创建，`is_leaf == True`，梯度会在 `.backward()` 后存入 `.grad`。
**非叶节点**：由运算派生，`is_leaf == False`，默认不保留梯度（`.grad` 为 `None`），需要 `retain_grad()` 才能查看。

```
a (leaf)  b (leaf)
  \          /
   c = a * b   <- c.grad_fn = MulBackward0, c.is_leaf = False
       |
   L = c + a   <- L.grad_fn = AddBackward0
```

In [ ]:
a = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(3.0, requires_grad=True)
c = a * b
L = c + a

print('a.is_leaf:', a.is_leaf)   # True
print('c.is_leaf:', c.is_leaf)   # False
print('c.grad_fn:', c.grad_fn)   # MulBackward0
print('L.grad_fn:', L.grad_fn)   # AddBackward0

## 2. `.detach()` 与 `.grad` 的存储规则

`.detach()` 返回一个与原张量共享数据但**切断梯度流**的新张量——之后的运算不再记录到图中。
常见用途：把中间激活値转为纯 numpy 打印，或在 GAN 训练中固定某一路径。

`.grad` 仅为叶节点自动保留；对非叶节点必须在 `backward()` 之前调用 `retain_grad()`，否则 `.grad` 保持 `None`。

```python
# 切断示例
x = torch.tensor(1.0, requires_grad=True)
y = x * 2
z = y.detach() * 3   # z 不在图上
z.backward()          # 报错：z 没有 grad_fn
```

In [ ]:
x = torch.tensor(1.0, requires_grad=True)
y = x * 2          # 非叶节点
y.retain_grad()    # 开启非叶梯度保留
L2 = y * 3
L2.backward()

print('x.grad:', x.grad)   # 6.0  (dL2/dx = 3*2)
print('y.grad:', y.grad)   # 3.0  (dL2/dy = 3)

# detach 演示
y_np = y.detach().numpy()   # 不影响图
print('y_np (detach):', y_np)

## 3. 梯度累积（gradient accumulation）与 `zero_grad()`

每次调用 `.backward()`，PyTorch 会把新梯度**加到** `.grad` 上，而不是覆盖。
这是有意设计（支持梯度累积训练技巧），但在标准训练循环中会导致梯度错误地叠加。

规范做法：
```python
for batch in dataloader:
    optimizer.zero_grad()   # 清零上一步残留
    loss = model(batch)
    loss.backward()
    optimizer.step()
```

手写 `Value` 引擎同理：每次反向前需手动把所有节点的 `.grad` 归零。

In [ ]:
w = torch.tensor(1.0, requires_grad=True)

# 两次 backward，不清零
for i in range(2):
    L3 = w * 3.0
    L3.backward()
    print(f'第 {i+1} 次 backward 后 w.grad = {w.grad}')  # 3.0, 6.0

# 清零后再做
w.grad.zero_()
L3 = w * 3.0
L3.backward()
print('zero_grad 后 w.grad =', w.grad)  # 3.0

## 3.5 `retain_graph=True`：保留计算图，允许多次 `backward()`

`retain_graph=True` 与 `retain_grad()` 是两个不同的 API，不要混淆：

| API | 保留的是什么 | 典型用途 |
|---|---|---|
| `tensor.retain_grad()` | 非叶节点的 `.grad` **值** | 调试中间层梯度 |
| `backward(retain_graph=True)` | **计算图结构**（DAG） | 同一前向图多次调用 `backward()` |

**为什么默认会释放图？**  
反向传播后，`grad_fn` 持有的 `saved_tensors` 不再需要，PyTorch 默认释放以节省内存。  
传入 `retain_graph=True` 后这些缓冲区被保留，直到下次 `backward()` 才释放。

**典型场景：多任务学习**
```python
shared_out = encoder(x)
loss1 = head1(shared_out).mean()
loss2 = head2(shared_out).mean()
loss1.backward(retain_graph=True)  # 保留图
loss2.backward()                   # 图在此释放
```


In [ ]:
# === 演示 retain_graph=True ===

w = torch.tensor(2.0, requires_grad=True)
x_fixed = torch.tensor(3.0)

# 同一前向传播，产生两个 loss
z = w * x_fixed           # z = 6.0
loss1 = z * z             # L1 = z²,  dL1/dw = 2z·x = 36.0
loss2 = z + 1.0           # L2 = z+1, dL2/dw = x   =  3.0

# 第一次 backward：保留图
loss1.backward(retain_graph=True)
print(f'loss1.backward(retain_graph=True) → w.grad = {w.grad.item()}')
assert abs(w.grad.item() - 36.0) < 1e-6, '期望 dL1/dw = 36.0'

# 第二次 backward：不再保留，图在此释放
w.grad.zero_()
loss2.backward()
print(f'loss2.backward()                 → w.grad = {w.grad.item()}')
assert abs(w.grad.item() - 3.0) < 1e-6, '期望 dL2/dw = 3.0'

# 对比：不加 retain_graph，提前释放图 → 第二次 backward 报 RuntimeError
print()
w2 = torch.tensor(2.0, requires_grad=True)
z2 = w2 * x_fixed
loss1b = z2 * z2
loss2b = z2 + 1.0
loss1b.backward()  # retain_graph=False（默认），图在此释放
try:
    loss2b.backward()
    print('错误：此处应抛出 RuntimeError')
except RuntimeError as e:
    print(f'✅ 预期 RuntimeError（图已释放）：{str(e)[:90]}')

print('\n结论：retain_graph=True 保留图；retain_grad() 保留非叶 .grad 值。两者正交，可同时使用。')


## 4. ✏️ 实现 `verify_gradients()`

**任务**：对 `L = 2*a*b + 3*b`（`a=2.0, b=3.0`），验证自动梯度和手算梯度一致。

**手算结果**：
```
∂L/∂a = 2b = 6.0
∂L/∂b = 2a + 3 = 7.0
```

**实现要点**：

| 步骤 | 操作 | 注意 |
|---|---|---|
| 1 | `a = torch.tensor(2.0, requires_grad=True)` | 必须 `requires_grad=True` |
| 2 | `L = 2*a*b + 3*b` | 每步都是 Tensor 运算 |
| 3 | `L.backward()` | 标量才能直接 backward() |
| 4 | `assert abs(a.grad - 6.0) < 1e-6` | 检验 a 的梯度 |
| 5 | `assert abs(b.grad - 7.0) < 1e-6` | 检验 b 的梯度 |

**验收标准**：函数运行无错，打印 `a.grad=6.0, b.grad=7.0`。

In [ ]:
def verify_gradients():
    raise NotImplementedError("TODO: 创建a=2.0,b=3.0,L=2*a*b+3*b,backward,assert grads")

verify_gradients()

In [ ]:
# --- 检查：调用学生实现的 verify_gradients() ---
import torch

def _ref_verify():
    a = torch.tensor(2.0, requires_grad=True)
    b = torch.tensor(3.0, requires_grad=True)
    L = a * b * b + a
    L.backward()
    return a.grad.item(), b.grad.item()

try:
    result = verify_gradients()
except NotImplementedError:
    result = None
if result is None:
    print("⬜ 请先实现 verify_gradients()，再运行此格")
    print("   参考答案：dL/da = b²+1 = 10，dL/db = 2ab = 12")
else:
    da, db = result
    assert abs(da - 10.0) < 1e-4, f'dL/da 应为 10.0（b²+1），实际 {da}'
    assert abs(db - 12.0) < 1e-4, f'dL/db 应为 12.0（2ab），实际 {db}'
    print('✅ dL/da =', da, '| dL/db =', db, '— 与解析値一致')


## 5. 参数实验：自制 Value vs torch 对 `L = tanh(w*x + b)` 的梯度误差

**实验参数**：`w` 在 `[-2, -1, 0, 1, 2]` 上扫描，`x=1.0, b=0.5` 固定。

**预期现象**：
- `atol` 级别误差应 < 1e-6（浮点精度以内）
- `tanh` 饱和区（`|w|` 大）梯度趋近 0，两者差値绝对値也趋近 0
- 图中两条曲线（torch / Value）完全重叠，误差曲线几乎贴近 x 轴

In [ ]:
import matplotlib.pyplot as plt
import math

# ---- 手写 Value（最小实现，与 L56_backward_pass.ipynb 一致）----
class Value:
    def __init__(self, data, _children=(), _op=''):
        self.data = float(data)
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        def _back():
            self.grad  += other.data * out.grad
            other.grad += self.data  * out.grad
        out._backward = _back
        return out

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        def _back():
            self.grad  += out.grad
            other.grad += out.grad
        out._backward = _back
        return out

    def tanh(self):
        t = math.tanh(self.data)
        out = Value(t, (self,), 'tanh')
        def _back():
            self.grad += (1 - t**2) * out.grad
        out._backward = _back
        return out

    def backward(self):
        topo, visited = [], set()
        def build(v):
            if v not in visited:
                visited.add(v)
                for c in v._prev: build(c)
                topo.append(v)
        build(self)
        self.grad = 1.0
        for v in reversed(topo): v._backward()

# ---- 对比实验 ----
ws = [-2.0, -1.0, 0.0, 1.0, 2.0]
x_val, b_val = 1.0, 0.5

grads_torch, grads_value, errors = [], [], []

for w_val in ws:
    # torch
    wt = torch.tensor(w_val, requires_grad=True)
    xt = torch.tensor(x_val)
    bt = torch.tensor(b_val)
    Lt = torch.tanh(wt * xt + bt)
    Lt.backward()
    gt = wt.grad.item()
    grads_torch.append(gt)

    # Value
    wv = Value(w_val)
    xv = Value(x_val)
    bv = Value(b_val)
    Lv = (wv * xv + bv).tanh()
    Lv.backward()
    gv = wv.grad
    grads_value.append(gv)

    errors.append(abs(gt - gv))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

ax1.plot(ws, grads_torch, 'o-', label='torch autograd', linewidth=2)
ax1.plot(ws, grads_value, 's--', label='手写 Value', linewidth=2, alpha=0.7)
ax1.set_xlabel('w')
ax1.set_ylabel('dL/dw')
ax1.set_title('dL/dw for L=tanh(w*x+b)')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.semilogy(ws, [max(e, 1e-20) for e in errors], 'r^-', linewidth=2)
ax2.set_xlabel('w')
ax2.set_ylabel('|torch - Value| (log scale)')
ax2.set_title('梯度误差（atol 级别）')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('最大误差 atol:', max(errors))

## 本课收束

`verify_gradients()` 输出了 `dL/da=10.0, dL/db=12.0`，与手写 `Value` 引擎及解析导数在 1e-6 精度内吻合。
本节确认：PyTorch `autograd` 的计算图遍历、叶节点梯度存储、以及 `zero_grad()` 规范，与 `L56_backward_pass.ipynb` 的纯 Python 实现在数学上等价。
这两套梯度引擎共同支撑 Aurora 的 `src/aurora/` 各核训练循环——Month 3 起的 `trainer.py` 将直接调用 `loss.backward()`。
下一节（L61）在此基础上构建第一个真正的线性层，用 `nn.Module` 封装参数并接入 `optimizer.step()`。

> **retain_graph vs retain_grad**：`retain_grad()` 保留非叶节点的 `.grad`（本节演示）；`loss.backward(retain_graph=True)` 保留计算图以便多次调用 `backward()`（用于多任务学习，后续多任务训练章节将演示此用法）。

## ✏️ 白板挑战：autograd 机制手推（目标 10 分钟）

盖上屏幕，纸上推导：

**问 1**：`L = 2*a*b + 3*b`，`a=2.0, b=3.0`。手算 `∂L/∂a` 和 `∂L/∂b`。  
（∂L/∂a=2b=6.0，∂L/∂b=2a+3=2×2+3=7.0）

**问 2**：为什么非叶节点的 `.grad` 默认是 `None`？用 `retain_grad()` 解决后有什么代价？  
（省显存；保留中间梯度会使内存增加，等同于保存所有激活值）

**问 3**：同一个 Tensor 做了两次 `backward()`（没有 `retain_graph=True`），会发生什么？  
（计算图在第一次 backward 后被释放；第二次会抛出 RuntimeError）

**问 4**：`w.grad.zero_()` 和 `w.grad = None` 有什么区别？  
（zero_() 保留 grad Tensor 对象，置零；grad=None 删除梯度引用；后者在 optimizer 里更常见）

**问 5**：`x.detach()` 后做运算，梯度还会流回 `x` 吗？为什么需要 detach？  
（不会流回；detach 切断梯度链，常用于将 logit 传给 loss 计算但不想影响某些参数）

推导完成后运行下方格验证。

In [ ]:
# ✏️ 对答案格
import torch

# 问1：手算梯度验证
a_q = torch.tensor(2.0, requires_grad=True)
b_q = torch.tensor(3.0, requires_grad=True)
L_q = 2*a_q*b_q + 3*b_q
L_q.backward()
assert abs(a_q.grad.item() - 6.0) < 1e-6, f"a.grad={a_q.grad.item()}"
assert abs(b_q.grad.item() - 7.0) < 1e-6, f"b.grad={b_q.grad.item()}"
print(f"Q1 ✅  L=2ab+3b: ∂L/∂a={a_q.grad.item():.1f}=2b=6, ∂L/∂b={b_q.grad.item():.1f}=2a+3=7")

# 问2：非叶节点grad=None，retain_grad保留
x_q = torch.tensor(2.0, requires_grad=True)
y_q = x_q * 3    # 非叶
y_q.retain_grad()
z_q = y_q ** 2
z_q.backward()
assert x_q.grad is not None      # 叶节点有grad
assert y_q.grad is not None      # retain_grad保留了中间grad
print(f"Q2 ✅  retain_grad后非叶grad={y_q.grad.item():.1f}，叶grad={x_q.grad.item():.1f}")

# 问3：二次backward报错
x2 = torch.tensor(1.0, requires_grad=True)
y2 = x2 ** 2
y2.backward()
try:
    y2.backward()
    print("Q3 ❌  应该报错但没有")
except RuntimeError as e:
    print(f"Q3 ✅  第二次backward报RuntimeError（图已释放）: {str(e)[:50]}...")

# 问4：zero_() vs grad=None
w_q = torch.tensor(1.0, requires_grad=True)
(w_q * 2).backward()
grad_obj_before = id(w_q.grad)
w_q.grad.zero_()
assert id(w_q.grad) == grad_obj_before   # 同一对象
w_q.grad = None
assert w_q.grad is None
print(f"Q4 ✅  zero_()保留grad对象，grad=None删除引用")

# 问5：detach切断梯度
x5 = torch.tensor(3.0, requires_grad=True)
xd = x5.detach()
y5 = xd * 2
# y5没有grad_fn
assert y5.grad_fn is None
print(f"Q5 ✅  x.detach()后: y=xd*2的grad_fn={y5.grad_fn}（None，梯度不会流回x）")
print("\n🎉 autograd 机制白板挑战通过！")

In [ ]:
# ✏️ 本课自评
l60_review = {
    "leaf_vs_nonleaf":         None,  # 理解叶/非叶节点区别，为什么非叶.grad=None？True/False
    "verify_gradients_impl":   None,  # verify_gradients()实现正确，梯度断言通过？True/False
    "detach_understood":       None,  # 理解detach()切断梯度流的场景和原因？True/False
    "grad_accumulation":       None,  # 理解不清零时多次backward梯度累积问题？True/False
    "whiteboard_passed":       None,  # 白板挑战5道手推全部完成？True/False
}

unfilled = [k for k, v in l60_review.items() if v is None]
assert not unfilled, f'还未填写：{unfilled}'
weak = [k for k, v in l60_review.items() if v is False]
if weak:
    print(f'⚠️  需要加强：{weak}')
else:
    print('✅ L60 全部通关！进入 L61：nn.Module 实战')

In [ ]:
# Aurora 连接：aurora.llm 使用 PyTorch autograd 构建训练循环
from aurora.llm.kvcache import KVCache  # aurora.llm 使用 PyTorch autograd
print("aurora.llm.kvcache.KVCache:", KVCache)

---

→ **下一课**　[L61 · nn.Module 实战](L61_pytorch_nn.ipynb)

> 下节课将学习 **nn.Module 实战**：Linear / ReLU / Sequential，参数管理与模型保存。